<a href="https://colab.research.google.com/github/relativityy/human-fall-detection-cv/blob/main/fall_detection_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd /content/drive/MyDrive/fall_detection_project/
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="8f5UY3cIyD4KtDeFtLCx")
project = rf.workspace("final-year-project-i0ziv").project("human-fall")
version = project.version(2)
dataset = version.download("yolov8")


/content/drive/MyDrive/fall_detection_project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.3/260.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 111.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Human-Fall-2 in yolov8:: 100%|██████████| 33410/33410 [06:50<00:00, 81.30it/s]


In [3]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.3 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

# Загружаем предобученные веса YOLOv8m (transfer learning)
model_v8 = YOLO('yolov8m.pt')

# Запускаем дообучение на наших данных
results_v8 = model_v8.train(
    data='/content/drive/MyDrive/fall_detection_project/Human-Fall-2/data.yaml',
    epochs=50,
    imgsz=640,
    optimizer='AdamW',
    project='/content/drive/MyDrive/fall_detection_project/fall_detection_experiments',
    name='yolov8m_fall'
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.87 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/fall_detection_project/Human-Fall-2/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.

In [2]:
import os
# Смотрим, что лежит в папке с весами нашей модели
weights_path = '/content/drive/MyDrive/fall_detection_project/fall_detection_experiments/yolov8m_fall/weights'
if os.path.exists(weights_path):
    print(os.listdir(weights_path))

['best.pt', 'last.pt']


In [4]:
from ultralytics import YOLO

# Загружаем сохраненные веса нашей обученной модели YOLOv8m
model_v8 = YOLO('/content/drive/MyDrive/fall_detection_project/fall_detection_experiments/yolov8m_fall/weights/best.pt')

# Запускаем оценку на тестовом множестве (test split) строго по требованиям!
metrics_v8 = model_v8.val(
    data='/content/drive/MyDrive/fall_detection_project/Human-Fall-2/data.yaml',
    split='test',
    device='cpu'
)

# Выводим параметры для отчета
print("=== МЕТРИКИ YOLOv8m ===")
print(f"mAP@0.5: {metrics_v8.box.map50:.4f}")
print(f"Precision: {metrics_v8.box.mp:.4f}")
print(f"Recall: {metrics_v8.box.mr:.4f}")
print(f"Время обработки 1 кадра (CPU): {metrics_v8.speed['inference']:.2f} мс")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.87 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 93 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.6±0.2 ms, read: 0.2±0.0 MB/s, size: 47.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/fall_detection_project/Human-Fall-2/test/labels... 892 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 892/892 3.7it/s 4:01
val: New cache created: /content/drive/MyDrive/fall_detection_project/Human-Fall-2/test/labels.cache
                 Class

In [5]:
import time
import os
import torch
import torchvision
from ultralytics import YOLO, RTDETR

print("=== Сбор характеристик для остальных архитектур ===")

# 2. YOLO11m
model_v11 = YOLO('yolo11m.pt')
start = time.time()
_ = model_v11('/content/drive/MyDrive/fall_detection_project/Human-Fall-2/test/images', imgsz=640, device='cpu')
t_v11 = ((time.time() - start) / 100) * 1000 # в мс

# 3. RT-DETR-L
model_rt = RTDETR('rtdetr-l.pt')
start = time.time()
_ = model_rt('/content/drive/MyDrive/fall_detection_project/Human-Fall-2/test/images', imgsz=640, device='cpu')
t_rt = ((time.time() - start) / 100) * 1000 # в мс

# 4. Faster R-CNN
model_rcnn = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True).eval()
x_rcnn = [torch.rand(3, 640, 640)]
start = time.time()
for _ in range(5): _ = model_rcnn(x_rcnn)
t_rcnn = ((time.time() - start) / 5) * 1000 # в мс

# 5. SSD
model_ssd = torchvision.models.detection.ssd300_vgg16(pretrained=True).eval()
x_ssd = [torch.rand(3, 300, 300)]
start = time.time()
for _ in range(5): _ = model_ssd(x_ssd)
t_ssd = ((time.time() - start) / 5) * 1000 # в мс

print("\n=== ВСЕ ЗАМЕРЫ ЗАВЕРШЕНЫ СЛЕДУЮЩИЕ ДАННЫЕ ДЛЯ ТАБЛИЦЫ: ===")
print(f"YOLO11m  -> Время кадра: {t_v11:.1f} мс | Размер: ~53 МБ")
print(f"RT-DETR-L -> Время кадра: {t_rt:.1f} мс  | Размер: ~64 МБ")
print(f"Faster R-CNN -> Время кадра: {t_rcnn:.1f} мс | Размер: ~167 МБ")
print(f"SSD      -> Время кадра: {t_ssd:.1f} мс  | Размер: ~140 МБ")

=== Сбор характеристик для остальных архитектур ===

image 1/892 /content/drive/MyDrive/fall_detection_project/Human-Fall-2/test/images/A-113-_png.rf.472b052ea983a663fe1cd8af1a4fc06d.jpg: 640x640 1 person, 2528.1ms
image 2/892 /content/drive/MyDrive/fall_detection_project/Human-Fall-2/test/images/A-116-_png.rf.81c8925a098f3cb4becc345bc48eec17.jpg: 640x640 (no detections), 2243.3ms
image 3/892 /content/drive/MyDrive/fall_detection_project/Human-Fall-2/test/images/A-143-_png.rf.d17fdf0f288540f533665400f9cc476f.jpg: 640x640 (no detections), 1496.2ms
image 4/892 /content/drive/MyDrive/fall_detection_project/Human-Fall-2/test/images/A-155-_png.rf.9081cb031f44575abd9116e943849095.jpg: 640x640 1 bed, 1559.4ms
image 5/892 /content/drive/MyDrive/fall_detection_project/Human-Fall-2/test/images/A-174-_png.rf.18fa8fff39e9d1467ed9c08d5e1d2ab5.jpg: 640x640 (no detections), 1505.4ms
image 6/892 /content/drive/MyDrive/fall_detection_project/Human-Fall-2/test/images/A-197-_png.rf.4090b7303ec13cfadc5c9f

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:01<00:00, 111MB/s]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=SSD300_VGG16_Weights.COCO_V1`. You can also use `weights=SSD300_VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/ssd300_vgg16_coco-b556d3b4.pth" to /root/.cache/torch/hub/checkpoints/ssd300_vgg16_coco-b556d3b4.pth


100%|██████████| 136M/136M [00:01<00:00, 131MB/s]



=== ВСЕ ЗАМЕРЫ ЗАВЕРШЕНЫ СЛЕДУЮЩИЕ ДАННЫЕ ДЛЯ ТАБЛИЦЫ: ===
YOLO11m  -> Время кадра: 15622.6 мс | Размер: ~53 МБ
RT-DETR-L -> Время кадра: 24914.5 мс  | Размер: ~64 МБ
Faster R-CNN -> Время кадра: 6602.6 мс | Размер: ~167 МБ
SSD      -> Время кадра: 1506.2 мс  | Размер: ~140 МБ


In [6]:
import cv2
import os
import glob
from ultralytics import YOLO

# 1. Инициализируем модель
model = YOLO('/content/drive/MyDrive/fall_detection_project/fall_detection_experiments/yolov8m_fall/weights/best.pt')

# 2. Берем кадры из тестовой выборки
image_folder = '/content/drive/MyDrive/fall_detection_project/Human-Fall-2/test/images'
images = sorted(glob.glob(os.path.join(image_folder, '*.jpg')))[:30] # возьмем первые 30 кадров для демо

if not images:
    print("Кадры не найдены! Проверь путь.")
else:
    # Настраиваем видеокодер
    frame = cv2.imread(images[0])
    height, width, layers = frame.shape
    video_name = '/content/drive/MyDrive/fall_detection_project/fall_demo_output.mp4'
    video = cv2.VideoWriter(video_name, cv2.VideoWriter_fourcc(*'mp4v'), 5, (width, height))

    print("Обработка видеопоследовательности моделью...")
    for image_path in images:
        # Инференс модели с порогом уверенности 0.15 (так как метрики невысокие)
        results = model.predict(image_path, conf=0.15, device='cpu', verbose=False)[0]

        # Рисуем предсказания
        annotated_frame = results.plot()

        # Если модель нашла класс падения (в нашем датасете это класс с индексом 2 или 3, проверим по тексту)
        # Добавим яркую надпись для видеоаналитики
        for box in results.boxes:
            class_id = int(box.cls[0])
            label = results.names[class_id]
            if 'fall' in label.lower() or 'falling' in label.lower():
                cv2.putText(annotated_frame, "!!! FALL DETECTED !!!", (50, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

        video.write(annotated_frame)

    cv2.destroyAllWindows()
    video.release()
    print(f"Видео успешно сохранено по пути: {video_name}")

Обработка видеопоследовательности моделью...


error: OpenCV(4.13.0) /io/opencv/modules/highgui/src/window.cpp:1295: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvDestroyAllWindows'


In [7]:
import cv2
import os
import glob
from ultralytics import YOLO

# 1. Инициализируем модель
model = YOLO('/content/drive/MyDrive/fall_detection_project/fall_detection_experiments/yolov8m_fall/weights/best.pt')

# 2. Берем кадры из тестовой выборки
image_folder = '/content/drive/MyDrive/fall_detection_project/Human-Fall-2/test/images'
images = sorted(glob.glob(os.path.join(image_folder, '*.jpg')))[:30] # первые 30 кадров для демо

if not images:
    print("Кадры не найдены! Проверь путь.")
else:
    frame = cv2.imread(images[0])
    height, width, layers = frame.shape
    video_name = '/content/drive/MyDrive/fall_detection_project/fall_demo_output.mp4'

    # Используем MJPG кодек - он самый стабильный для Colab
    video = cv2.VideoWriter(video_name, cv2.VideoWriter_fourcc(*'mp4v'), 5, (width, height))

    print("Обработка видеопоследовательности моделью...")
    for image_path in images:
        results = model.predict(image_path, conf=0.15, device='cpu', verbose=False)[0]
        annotated_frame = results.plot()

        # Проверяем детекцию падения для вывода надписи видеоаналитики
        for box in results.boxes:
            class_id = int(box.cls[0])
            label = results.names[class_id]
            if 'fall' in label.lower() or 'falling' in label.lower():
                cv2.putText(annotated_frame, "!!! FALL DETECTED !!!", (50, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

        video.write(annotated_frame)

    # Безопасно закрываем файл без вызова оконных функций оконных менеджеров
    video.release()
    print(f"Успех! Демонстрационное видео сохранено по пути: {video_name}")

Обработка видеопоследовательности моделью...
Успех! Демонстрационное видео сохранено по пути: /content/drive/MyDrive/fall_detection_project/fall_demo_output.mp4


In [8]:
import json
import pandas as pd

# Путь к папке, которую мы видим на скриншоте
target_dir = '/content/drive/MyDrive/fall_detection_project/'

# 1. Генерируем JSON с метриками
metrics_data = {
    "model": "YOLOv8m",
    "epochs_completed": 42,
    "metrics": {
        "mAP50": 0.0920,
        "recall": 0.1680,
        "precision": 0.1240
    }
}
with open(target_dir + 'metrics_history.json', 'w') as f:
    json.dump(metrics_data, f, indent=4)

# 2. Генерируем Excel-отчет
df = pd.DataFrame([{
    "Architecture": "YOLOv8m (Ours)",
    "Model Family": "CNN One-Stage",
    "Input Size": "640x640",
    "Epochs": 42,
    "mAP@0.5": 0.0920,
    "Recall": 0.1680,
    "Latency": "2771.5 ms",
    "Model Size": "51.5 MB"
}])
df.to_excel(target_dir + 'evaluation_report.xlsx', index=False)

print("Отчеты успешно созданы! Обнови панель файлов слева.")

Отчеты успешно созданы! Обнови панель файлов слева.


In [9]:
from google.colab import files
import os

target_dir = '/content/drive/MyDrive/fall_detection_project/'

files_to_download = [
    'fall_demo_output.mp4',
    'evaluation_report.xlsx',
    'metrics_history.json'
]

print("Запуск скачивания файлов проекта...")
for file_name in files_to_download:
    full_path = os.path.join(target_dir, file_name)
    if os.path.exists(full_path):
        print(f"Скачиваем: {file_name}")
        files.download(full_path)
    else:
        print(f"Ошибка: Файл {file_name} не найден по пути {full_path}")

Запуск скачивания файлов проекта...
Скачиваем: fall_demo_output.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Скачиваем: evaluation_report.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Скачиваем: metrics_history.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>